# 03 DQN：如何用 TD 一步步修正动作评分

这一课只回答一个问题：**DQN 如何在游戏进行中，用一条刚发生的经验修正它对动作长期价值的预测？** 读完后应能说清 DQN 的输入输出、TD 目标，以及“行动一次和训练一次”在时间上如何衔接。


## 1. DQN 要学什么，网络又输出什么？

最优动作价值函数 $Q_\star(s,a)$ 表示：在状态 $s$ 先执行动作 $a$，之后采取最优行动，能得到的最大**期望**累计回报。它像“先知”，但不是预测某一局所有随机细节；它只给动作的长期平均好坏打分。

DQN 用神经网络近似它：

$$
Q(s,a;w)\approx Q_\star(s,a).
$$

对离散动作，动作集合预先约定在网络外。例如 $\mathcal{A}=\{\text{左},\text{右},\text{上}\}$。网络输入状态表示 $s$，输出的是评分向量，不是动作本身：

$$
s\longrightarrow\operatorname{DQN}\longrightarrow
\begin{bmatrix}
Q(s,\text{左})\\
Q(s,\text{右})\\
Q(s,\text{上})
\end{bmatrix}.
$$

若输出为 $[370,-21,610]$，三个位置分别对应左、右、上跳；程序通常选最高分动作：

$$
a_t=\underset{a\in\mathcal{A}}{\operatorname{argmax}}\ Q(s_t,a;w).
$$


## 2. 每个步骤在什么时候发生？

最基础的一轮交互与训练按下面顺序发生：

$$
s_t
\longrightarrow
a_t
\longrightarrow
(r_{t+1},s_{t+1})
\longrightarrow
\text{训练一次}
\longrightarrow
a_{t+1}.
$$

先观察状态 $s_t$；网络给所有动作评分，并选择 $a_t$。执行动作后，环境立刻返回即时奖励 $r_{t+1}$ 和下一状态 $s_{t+1}$。于是得到一条经验：

$$
(s_t,a_t,r_{t+1},s_{t+1}).
$$

接着用这条经验训练网络，再将 $s_{t+1}$ 作为新的当前状态，继续选择下一动作。实际 DQN 常先把经验放进经验回放池，随机抽一小批旧经验训练；但单条经验的时间顺序不变。


## 3. TD 目标：一步事实加上对后续的估计

当前动作的完整长期回报未知，但它可以拆成“当前一步的奖励”和“之后的回报”。DQN 用 TD 目标作为当前评分的参考答案：

$$
\hat y_t=r_{t+1}+\gamma\max_{a\in\mathcal{A}}Q(s_{t+1},a;w).
$$

其中 $r_{t+1}$ 是刚刚真实观察到的即时奖励；第二项是网络对下一状态最佳未来价值的当前估计。TD 不是只看即时奖励，而是：

$$
\text{一步真实奖励}
+
\text{下一状态的未来估计}.
$$

因此不用等整局游戏结束，就能开始学习。


## 4. 网络怎样修正评分？

网络原先对刚执行动作的预测为：

$$
\hat q_t=Q(s_t,a_t;w).
$$

将它与 TD 目标比较：

$$
\delta_t=\hat q_t-\hat y_t.
$$

这叫 TD 误差。若它小于零，网络低估了，应调高评分；若大于零，网络高估了，应调低。训练使用平方损失：

$$
L(w)=\frac{1}{2}\bigl(\hat q_t-\hat y_t\bigr)^2.
$$

反向传播和梯度下降会调整网络参数：

$$
w\leftarrow w-\alpha\,\delta_t\,\nabla_wQ(s_t,a_t;w).
$$

人话：网络把“旧评分”与“基于新经验算出的参考答案”比较，然后调内部参数，让下一次评分更接近参考答案。


## 5. 一个数字例子

玛丽在 $s_t$ 选择上跳。网络原先估分为 $\hat q_t=70$。执行后，实际吃到金币，$r_{t+1}=1$；到达下一状态，网络估计从那里开始的最佳未来价值为 $100$。取 $\gamma=0.9$：

$$
\hat y_t=1+0.9\times100=91.
$$

因此：

$$
\delta_t=70-91=-21.
$$

负数说明低估。训练不会一下把 $70$ 改成 $91$，而是把参数调一点，让下次相似情况的预测向 $91$ 靠近。若预测是 $110$，误差为正，就应把评分往下调。


## 6. TD 与蒙特卡洛的区别

| 方法 | 当前动作价值的参考答案 | 是否等待回合结束？ |
| --- | --- | --- |
| 蒙特卡洛 | 此后直到终点的实际完整回报 | 是 |
| TD / DQN | 一步实际奖励 $+$ 下一状态的当前价值估计 | 否 |

终点附近的真实奖励会先让终点附近的估计变准，再逐步向更早的状态传播。这就是 TD 能在游戏尚未结束时学习的原因。


## 7. 小结与自检

- DQN 输出所有离散动作的评分向量，外部规则选择最高分动作。
- 一步经验为 $(s_t,a_t,r_{t+1},s_{t+1})$。
- TD 目标是“即时奖励 $+$ 下一状态最佳未来估计”。
- 训练让网络预测的 $Q(s_t,a_t;w)$ 靠近 TD 目标。

自检：网络预测为 $110$，即时奖励为 $1$，$\gamma=0.9$，下一状态的最佳评分为 $100$。TD 目标和 TD 误差分别是多少？评分应向哪个方向修正？

答：目标为 $1+0.9\times100=91$，误差为 $110-91=19$。网络高估，应向下修正当前状态动作的评分。
